# Project Main File

In [5]:
##### DOWNLOAD DATA FROM WEB #####

import kagglehub
path = kagglehub.dataset_download("iamsouravbanerjee/cause-of-deaths-around-the-world")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/iamsouravbanerjee/cause-of-deaths-around-the-world/versions/3


In [6]:
##### DATAFRAME IMPORT CELL #####

import pandas as pd

# Disease DataFrame.
filename = "cause_of_deaths.csv"
df = pd.read_csv(path + "/" + filename)

# HDI DataFrame -- manually uploaded to the Jupyter server.
df_hdi_raw = pd.read_csv("Human Development Index - Full.csv")

In [7]:
##### VIEW RAW DATA #####

df.head()

,Country/Territory,Code,Year,Meningitis,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Nutritional Deficiencies,Malaria,Drowning,Interpersonal Violence,...,Diabetes Mellitus,Chronic Kidney Disease,Poisonings,Protein-Energy Malnutrition,Road Injuries,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,Digestive Diseases,"Fire, Heat, and Hot Substances",Acute Hepatitis
0,Afghanistan,AFG,1990,2159,1116,371,2087,93,1370,1538,...,2108,3709,338,2054,4154,5945,2673,5005,323,2985
1,Afghanistan,AFG,1991,2218,1136,374,2153,189,1391,2001,...,2120,3724,351,2119,4472,6050,2728,5120,332,3092
2,Afghanistan,AFG,1992,2475,1162,378,2441,239,1514,2299,...,2153,3776,386,2404,5106,6223,2830,5335,360,3325
3,Afghanistan,AFG,1993,2812,1187,384,2837,108,1687,2589,...,2195,3862,425,2797,5681,6445,2943,5568,396,3601
4,Afghanistan,AFG,1994,3027,1211,391,3081,211,1809,2849,...,2231,3932,451,3038,6001,6664,3027,5739,420,3816


In [8]:
df_hdi_raw.head()

,ISO3,Country,Human Development Groups,UNDP Developing Regions,HDI Rank (2021),Human Development Index (1990),Human Development Index (1991),Human Development Index (1992),Human Development Index (1993),Human Development Index (1994),...,Material footprint per capita (tonnes) (2012),Material footprint per capita (tonnes) (2013),Material footprint per capita (tonnes) (2014),Material footprint per capita (tonnes) (2015),Material footprint per capita (tonnes) (2016),Material footprint per capita (tonnes) (2017),Material footprint per capita (tonnes) (2018),Material footprint per capita (tonnes) (2019),Material footprint per capita (tonnes) (2020),Material footprint per capita (tonnes) (2021)
0,AFG,Afghanistan,Low,SA,180.0,0.273,0.279,0.287,0.297,0.292,...,1.86,1.88,1.66,1.62,1.66,1.41,1.32,1.38,1.38,1.38
1,AGO,Angola,Medium,SSA,148.0,NaN,NaN,NaN,NaN,NaN,...,4.09,4.53,3.97,3.59,2.79,2.64,2.28,2.18,2.18,2.18
2,ALB,Albania,High,ECA,67.0,0.647,0.629,0.614,0.617,0.624,...,12.44,11.49,13.14,12.61,14.39,14.46,12.85,12.96,12.96,12.96
3,AND,Andorra,Very High,NaN,40.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARE,United Arab Emirates,Very High,AS,26.0,0.728,0.739,0.742,0.748,0.755,...,49.56,49.68,55.49,59.76,64.95,75.61,65.97,68.95,68.95,68.95


In [4]:
# INITIAL INSIGHTS

print(f"There are {len(df['Country/Territory'].unique())} countries in the cause of deaths dataset", end="\n\n")
print(f"The dataset spans the following years: {df['Year'].unique()}")

There are 204 countries in the cause of deaths dataset

The dataset spans the following years: [1990 1991 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003
 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017
 2018 2019]


## Cleaning up `chronic_df`

In [5]:
chronic_causes = ["Alzheimer's Disease and Other Dementias",
                  "Parkinson's Disease", 'Cardiovascular Diseases',
                  'Neoplasms', 'Diabetes Mellitus', 'Chronic Kidney Disease',
                  'Chronic Respiratory Diseases', 'Cirrhosis and Other Chronic Liver Diseases', 'HIV/AIDS']

chronic_df = df[['Country/Territory', 'Code', 'Year'] + chronic_causes].copy()

In [6]:
##### AGGREGATING ALL CHRONIC DEATHS PER COUNTRY PER YEAR #####

chronic_df['Total Chronic Deaths'] = chronic_df[chronic_causes].sum(axis = 1)

chronic_df.head(3)

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345


In [7]:
##### NORMALIZING DATAFRAME #####

normDf = pd.concat(
    [
        chronic_df[chronic_causes].div(chronic_df[chronic_causes].sum(axis = 1), axis = 0),
        chronic_df[['Country/Territory', 'Code', 'Year', 'Total Chronic Deaths']]
    ],
    axis = 1
)[
    ['Country/Territory', 'Code', 'Year'] +
    chronic_causes +
    ['Total Chronic Deaths']
]

In [8]:
normDf.head()

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths
0,Afghanistan,AFG,1990,0.015407,0.005122,0.619852,0.159867,0.029102,0.051205,0.082074,0.036902,0.000469,72435
1,Afghanistan,AFG,1991,0.015464,0.005091,0.619267,0.160575,0.028859,0.050694,0.082357,0.037135,0.000558,73461
2,Afghanistan,AFG,1992,0.015422,0.005017,0.617918,0.162161,0.028575,0.050116,0.082593,0.037561,0.000637,75345
3,Afghanistan,AFG,1993,0.015285,0.004945,0.617472,0.162690,0.028265,0.049732,0.082993,0.037897,0.000721,77657
4,Afghanistan,AFG,1994,0.015187,0.004903,0.618352,0.161949,0.027978,0.049310,0.083571,0.037960,0.000790,79741


In [9]:
##### AGGREGATING DATA BY COUNTRY #####

normDf_mean = normDf.groupby(['Country/Territory', 'Code'])[chronic_causes + ['Total Chronic Deaths']].mean()
normDf_mean.head(3)

,,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths
Country/Territory,Code,,,,,,,,,,
Afghanistan,AFG,0.015665,0.005004,0.602641,0.174575,0.034372,0.050301,0.079014,0.036889,0.001539,89082.966667
Albania,ALB,0.037107,0.010014,0.615455,0.234914,0.009396,0.017761,0.054609,0.020611,0.000134,14577.233333
Algeria,DZA,0.026061,0.006966,0.664508,0.144614,0.026801,0.047496,0.052858,0.028899,0.001797,107104.600000


## Cleaning up df_hdi_raw

In [11]:
pd.set_option('display.max_columns', None)

selected_columns = [
    col for col in df_hdi_raw.columns
    if any(
        phrase.lower() in col.lower()
        for phrase in [
            "Gross National Income",
            "Life Expectancy",
            "Human Development Index",
            "ISO3",
            "Country",
            "Human Development Groups"
        ]
    )
] #choosing our parameters (HDI, Life exp, human dev) 

df_selected_raw = df_hdi_raw[selected_columns]
df_hdi = df_selected_raw.drop(columns=[col for col in df_selected_raw.columns if 'male' in col.lower() or 'inequality' in col.lower() or 'Planetary pressures-adjusted' in col])

In [1]:
df_hdi.head()

NameError: name 'df_hdi' is not defined

In [12]:
hdi_long = df_hdi.melt(
    id_vars=['Country'],
    var_name='Metric_Year',
    value_name='Value'
)

hdi_long['Metric'] = hdi_long['Metric_Year'].str.extract(r'^(.*) \(')
year_extracted = hdi_long['Metric_Year'].str.extract(r'\((\d{4})\)')
hdi_long['Year'] = pd.to_numeric(year_extracted[0], errors='coerce').astype('Int64')  

hdi_pivot = hdi_long.pivot_table(
    index=['Country', 'Year'],
    columns='Metric',
    values='Value'
).reset_index()

hdi_pivot.columns.name = None
hdi_pivot = hdi_pivot.rename(columns={
    'Human Development Index': 'HDI',
    'Life Expectancy at Birth': 'LifeExpectancy',
    'Gross National Income Per Capita': 'GNI_per_capita'
})

merged_df = pd.merge(
    chronic_df,
    hdi_pivot,
    how='inner',
    left_on=['Country/Territory', 'Year'],
    right_on=['Country', 'Year']
)

merged_df = merged_df.drop(columns=['Country'])
merged_df.head(4) #


,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664


## The DataFrame `merged_df` has the information from all countries through all years with all chronic diseases and the metrics HDI, LifeExp, GNI.

We will now proceed to answer the questions of interest:

- Can the perfect ratio discussed above be used to predict the rank/development index of a country? 
- Does the perfect ratio compare well with the human development index?
- Does the perfect ratio compare well with life expectancy? 
- Does the perfect ratio compare well with GNI per capita (gross national income)?
- Is the most probable chronic death dependent on region?
- What method of calculating the perfect ratio makes most sense? (Compare perfect ratios)
- Is there correlation between different chronic diseases? (independence/dependence)


# Question 3
How far is each country’s death ratio from an "ideal" death ratio? Does this distance correlate with development indicators like HDI?

The "ideal" death ratio could represent a theoretically healthy society—either defined normatively (e.g., low mortality across all diseases) or empirically (e.g., average of the healthiest countries).

The death ratio vector for all countries and development metrics (HDI, GNI per capita, life expectancy). To define an ideal vector, we’ll either use the mean of top-10 HDI countries or construct a hypothetical low-mortality ratio.

We’ll compute the Euclidean and cosine distances from each country’s death ratio to the ideal. We’ll then rank countries by closeness to the ideal and compare this ranking to HDI or GNI rankings using Spearman correlation.

We’ll report distance metrics and how well they correlate (Spearman ρ) with known development metrics. A strong correlation would suggest that the death ratio distance can serve as a proxy for health-related development.

In [13]:
merged_df

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664
4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.96189,0.292,51.4945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5515,Zimbabwe,ZWE,2015,754,215,16649,11161,3176,2108,2751,1956,29162,67932,3638.532892,0.582,59.5911
5516,Zimbabwe,ZWE,2016,767,219,16937,11465,3259,2160,2788,1962,27141,66698,3606.750671,0.588,60.3059
5517,Zimbabwe,ZWE,2017,781,223,17187,11744,3313,2196,2818,2007,24846,65115,3728.918785,0.594,60.7095
5518,Zimbabwe,ZWE,2018,795,227,17460,12038,3381,2240,2849,2030,22106,63126,3864.012419,0.602,61.4141


In [14]:
merged_df.to_csv('merged-df.csv')